# Tools (LangChain 1.3+)

Tools let a model **request actions** (search, DB lookup, run code) instead of only generating text. Each tool is:

| Part | What it is |
|------|------------|
| **Schema** | Name, description, argument types (JSON schema sent to the LLM) |
| **Function** | Python callable that runs when the model emits a `tool_call` |

**Prerequisite:** `GROQ_API_KEY` in repo-root `.env`. See **[2-modelintegration.ipynb](2-modelintegration.ipynb)** for model setup.

**Sections:** 1 baseline model | 2 `@tool` + `bind_tools` | 3 read `tool_calls` | 4 manual ReAct loop | 5 `create_agent` shortcut

Deep dive: **[reference/README.md](reference/README.md)** (ReAct loop, message trail).

In [ ]:
"""Load API keys from .env before any model or tool calls."""
import os
from pathlib import Path

from dotenv import load_dotenv

for env_path in (Path.cwd() / ".env", Path.cwd().parent / ".env"):
    if env_path.is_file():
        load_dotenv(env_path)
        break
else:
    load_dotenv()

for key in ("GROQ_API_KEY",):
    value = os.getenv(key)
    if value:
        os.environ[key] = value
    else:
        print(f"{key}: missing (add to .env)")

## 1. Model without tools

A plain chat model answers from training data only. `tool_calls` is empty and `finish_reason` is `stop`.

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model("groq:qwen/qwen3-32b")  # ChatGroq, reads GROQ_API_KEY

response = model.invoke("What is the capital of France?")
response  # AIMessage — no tool_calls

## 2. Define a tool and bind it to the model

- **`@tool`** — decorator from `langchain.tools`; docstring becomes the tool **description** in the schema
- **`model.bind_tools([...])`** — returns a new runnable that sends tool definitions to the API on every call
- The LLM **does not run** your Python function; it only **requests** a call via `tool_calls`

In [ ]:
from langchain.tools import tool


@tool
def get_weather(location: str) -> str:
    """Get the current weather for a given location."""
    return f"It's currently 70 degrees and cloudy in {location}"


# bind_tools attaches schemas — model can now emit tool_calls for get_weather
model_with_tools = model.bind_tools([get_weather])

## 3. Inspect `tool_calls` on the response

When the model decides to use a tool:

- `response.content` is often **empty** (action only, no final answer yet)
- `response.tool_calls` lists requested calls: `name`, `args`, `id`
- `finish_reason` is `tool_calls` (not `stop`)

You still need to **run** the tool and call the model again (section 4) — or use `create_agent` (section 5).

In [ ]:
ai_msg = model_with_tools.invoke("What is the weather in Tokyo?")
print(ai_msg)
print()

for tool_call in ai_msg.tool_calls:
    print(f"Tool name:  {tool_call['name']}")
    print(f"Tool args:  {tool_call['args']}")
    print(f"Tool call id: {tool_call['id']}")

## 4. Manual tool execution loop (mini ReAct)

This is what `create_agent` automates:

```
HumanMessage → AIMessage (tool_calls) → run tool → ToolMessage → AIMessage (final answer)
```

| Step | Who | What |
|------|-----|------|
| 1 | LLM | Chooses tool + args (`tool_calls`) |
| 2 | Your code | Runs `get_weather.invoke(args)` |
| 3 | You | Append `ToolMessage` with result + `tool_call_id` |
| 4 | LLM | Reads tool output, writes natural-language answer |

In [ ]:
# Step 1: user question → model requests tool call
messages = [{"role": "user", "content": "What is the weather in Tokyo?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: execute each tool call and append ToolMessage(s)
for tool_call in ai_msg.tool_calls:
    observation = get_weather.invoke(tool_call["args"])
    messages.append(
        {
            "role": "tool",
            "content": observation,
            "tool_call_id": tool_call["id"],
        }
    )

# Step 3: model sees tool result → final answer
final_response = model_with_tools.invoke(messages)
print(final_response.content)

## 5. Shortcut: `create_agent`

Instead of the manual loop, **`create_agent`** (LangGraph) wires `model` + `tools` nodes and repeats until there are no more `tool_calls`.

See **[1_lang_chain_intro.ipynb](1_lang_chain_intro.ipynb)** for the full agent walkthrough.

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model="groq:qwen/qwen3-32b",
    tools=[get_weather],
    system_prompt="You are a helpful assistant.",
)

result = agent.invoke(
    {"messages": [{"role": "user", "content": "What is the weather in Tokyo?"}]}
)
result["messages"][-1].content  # final AIMessage text

### Summary

| Approach | When to use |
|----------|-------------|
| `@tool` + `bind_tools` + manual loop | Learning ReAct, custom control over each step |
| `create_agent` | Production agents — automatic tool loop |

```python
@tool
def my_tool(x: str) -> str: ...

model_with_tools = model.bind_tools([my_tool])
ai_msg = model_with_tools.invoke("...")       # may have tool_calls
get_weather.invoke(ai_msg.tool_calls[0]["args"])  # run Python yourself

agent = create_agent(model=..., tools=[my_tool])  # loop handled for you
```